In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
import warnings

# Suprimir avisos
warnings.filterwarnings('ignore')

# 1. PREPARAÇÃO DOS DADOS
print("Carregando df_model.csv...")
try:
    df = pd.read_csv("df_model.csv")
except FileNotFoundError:
    print("Erro: Arquivo df_model.csv não encontrado.")
    exit()
    
# 2. TÉCNICA: TRANSFORMAÇÃO LOGARÍTMICA
# Criamos uma nova coluna 'Rating' usando log(1 + Quantity)
# Isso comprime a escala e reduz drasticamente o impacto de outliers.
print("Aplicando transformação logarítmica (log1p) na coluna 'Quantity'...")
df['Rating'] = np.log1p(df['Quantity'])

Carregando df_model.csv...
Aplicando transformação logarítmica (log1p) na coluna 'Quantity'...


In [2]:
# 3. FAZER O TREINAMENTO (COM DADOS TRANSFORMADOS)

# Técnica: Definir o 'Reader' para o Surprise
# Agora usamos a escala da nova coluna 'Rating'
min_rating = df['Rating'].min()
max_rating = df['Rating'].max()
print(f"Nova escala de 'Rating' (logarítmica): Min={min_rating:.2f}, Max={max_rating:.2f}")
reader = Reader(rating_scale=(min_rating, max_rating))

# Técnica: Carregar os dados no formato do Surprise
# Note que agora passamos a coluna 'Rating', e não mais 'Quantity'
data = Dataset.load_from_df(df[['CustomerID', 'StockCode', 'Rating']], reader)

# Técnica: Dividir os dados em treino e teste (75% treino, 25% teste)
# random_state=42 garante que a divisão seja sempre a mesma (reprodutibilidade)
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

# Técnica: Usar SVD (Filtragem Colaborativa)
model = SVD()

# Treinar o modelo
print("\nIniciando o treinamento do modelo (com dados transformados)...")
model.fit(trainset)
print("Treinamento concluído.")

Nova escala de 'Rating' (logarítmica): Min=0.69, Max=11.30

Iniciando o treinamento do modelo (com dados transformados)...
Treinamento concluído.


In [3]:
# 4. APLICANDO O MODELO
print("Aplicando o modelo ao conjunto de teste...")
predictions = model.test(testset)
print("Aplicação (teste) concluída.")

# 5. AVALIANDO O DESEMPENHO DO MODELO
print("Avaliando o desempenho (RMSE) na escala logarítmica...")
# Técnica: Cálculo do RMSE
# Este RMSE agora é na escala logarítmica 
# (ex: um erro de 0.5 significa log(real) - log(previsto))
rmse = accuracy.rmse(predictions)

print(f"\nResultado da Avaliação: O NOVO RMSE do modelo é: {rmse}")

# 6. VERIFICAR PREVISÃO
# Vamos usar o mesmo exemplo de antes.
# O modelo agora prevê o 'Rating' (logarítmico), não a 'Quantity'
uid_exemplo = 17850.0  
iid_exemplo = '85123A' 
pred = model.predict(uid=uid_exemplo, iid=iid_exemplo)

print(f"\n--- Exemplo de Previsão (Escala Logarítmica) ---")
print(f"Previsão para Usuário {uid_exemplo} e Item {iid_exemplo} (log): {pred.est:.4f}")

# Para interpretar, podemos reverter o log (usando expm1, que é e^x - 1)
# Esta é a função inversa do log1p(x)
real_quantity_pred = np.expm1(pred.est)
print(f"Isso equivale a uma 'Quantidade' prevista de aprox.: {real_quantity_pred:.0f} unidades.")

Aplicando o modelo ao conjunto de teste...
Aplicação (teste) concluída.
Avaliando o desempenho (RMSE) na escala logarítmica...
RMSE: 0.5196

Resultado da Avaliação: O NOVO RMSE do modelo é: 0.5196258443129227

--- Exemplo de Previsão (Escala Logarítmica) ---
Previsão para Usuário 17850.0 e Item 85123A (log): 2.1121
Isso equivale a uma 'Quantidade' prevista de aprox.: 7 unidades.
